# SVCS Final Results
**EGN 4950C Group 16 — Capstone Final Report**
Florida Atlantic University · Spring 2026 · Sponsored by NIWC Pacific / Defense Innovation Unit

This notebook aggregates and visualises the measured results from the SVCS (Selective Video Compression System) project for the May 6, 2026 capstone presentation.

Each cell is designed to run end-to-end without errors. External data sources (segments DB, super-resolution result JSON, CDnet sample clips) are optional — when they are missing, the notebook falls back to the values measured during the milestone runs and documented in `docs/final_report.md`.

Run cells top to bottom from a fresh kernel.

*Author: Bloodawn (KheivenD)*


In [ ]:
# Setup — import path, plot styling, results directory.
# This cell is the only hard prerequisite; later cells degrade gracefully
# when their optional data sources are missing.
import json
import sqlite3
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make sure src/ is importable when running from notebooks/
PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

plt.rcParams.update({
    'figure.dpi': 110,
    'font.family': 'sans-serif',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

RESULTS_DIR = PROJECT_ROOT / 'results'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
CDNET_DIR   = PROJECT_ROOT / 'data' / 'samples' / 'cdnet_mp4'
SAMPLES_DIR = PROJECT_ROOT / 'data' / 'samples'

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('Project root :', PROJECT_ROOT)
print('Results dir  :', RESULTS_DIR, '(exists)' if RESULTS_DIR.exists() else '(created)')
print('Outputs dir  :', OUTPUTS_DIR, '(exists)' if OUTPUTS_DIR.exists() else '(missing — DB cells will use fallback values)')
print('CDnet dir    :', CDNET_DIR,   '(exists)' if CDNET_DIR.exists()   else '(missing — side-by-side cell will use any local clip)')

---
## 1. Headline Numbers (from `docs/final_report.md`)

These are the canonical figures cited in the capstone report and the slide deck. The values were measured by `notebooks/milestone1_benchmark.ipynb` on the CDnet 2014 baseline clips and re-validated against the M3 stress-test runs (`docs/stress_test_results.md`).


In [ ]:
# Canonical headline numbers from docs/final_report.md, table in section 7.
# These come from real measurements — keep them in sync with the report.
HEADLINE = {
    'foreground_psnr_db'     : 41.2,
    'foreground_ssim'        : 0.9783,
    'background_psnr_db'     : 29.1,
    'background_ssim'        : 0.7903,
    'background_compression' : 16.6,    # x — Scenario 2 (no foreground detected)
    'effective_compression'  : 6.3,     # x — typical scene with ~5% foreground
    'foreground_compression' : 1.0,     # x — near-lossless intentionally
}

print('SVCS Headline Benchmark Numbers')
print('=' * 48)
for k, v in HEADLINE.items():
    label = k.replace('_', ' ').title()
    suffix = ' dB' if k.endswith('db') else ('x' if 'compression' in k else '')
    print(f'  {label:<28} {v}{suffix}')

---
## 2. Compression Ratio by Mode

Median compression ratio (vs. naive full-frame H.264) for each of the four pipeline modes. Range bars show the min / max ratio observed across the CDnet 2014 baseline category.


In [ ]:
# Source: M3 stress test runs across the CDnet 2014 baseline set.
# Mode 2/3 numbers come from the April 18-26 runs documented in
# docs/stress_test_results.md.
COMPRESSION_RESULTS = {
    'Mode 0\n(All frames)'  : (6.3, 3.8, 10.4),    # effective ratio (typical 5% FG)
    'Mode 1\n(Motion only)' : (9.3, 5.2, 16.1),
    'Mode 2\n(BG keyframe)' : (12.7, 6.9, 22.3),
    'Mode 3\n(Object only)' : (16.6, 9.1, 31.6),
}

modes  = list(COMPRESSION_RESULTS.keys())
medians = np.array([v[0] for v in COMPRESSION_RESULTS.values()])
mins    = np.array([v[1] for v in COMPRESSION_RESULTS.values()])
maxs    = np.array([v[2] for v in COMPRESSION_RESULTS.values()])

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(modes))
colors = ['#4e9af1', '#50c878', '#f4a261', '#e76f51']

ax.bar(x, medians, color=colors, alpha=0.85, width=0.55, zorder=3)
ax.errorbar(x, medians, yerr=[medians - mins, maxs - medians],
            fmt='none', color='#333', capsize=5, linewidth=1.5, zorder=4)

for i, v in enumerate(medians):
    ax.text(i, v + 0.4, f'{v:.1f}x', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(modes, fontsize=10)
ax.set_ylabel('Compression ratio (x vs. naive H.264)', fontsize=11)
ax.set_title('Storage compression by mode — CDnet 2014 baseline', fontsize=12)
ax.axhline(1, color='#999', linewidth=0.8, linestyle='--', label='No compression')
ax.set_ylim(0, maxs.max() * 1.15)
ax.grid(axis='y', alpha=0.3, zorder=0)
ax.legend(fontsize=9)

plt.tight_layout()
fig.savefig(RESULTS_DIR / 'fig_compression_ratio.png', bbox_inches='tight')
plt.show()
print('Saved:', RESULTS_DIR / 'fig_compression_ratio.png')

---
## 3. PSNR / SSIM — Foreground vs. Background

Quality is measured separately for ROI (foreground) regions encoded at CRF 18 and the surrounding background encoded at CRF 45. Foreground exceeds the project's PSNR ≥ 30 dB and SSIM ≥ 0.85 acceptance criteria. Background is intentionally degraded to maximise storage savings.


In [ ]:
quality = pd.DataFrame({
    'Region'         : ['Foreground (CRF 18)', 'Background (CRF 45)'],
    'PSNR (dB)'      : [HEADLINE['foreground_psnr_db'], HEADLINE['background_psnr_db']],
    'SSIM'           : [HEADLINE['foreground_ssim'],    HEADLINE['background_ssim']],
})

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
colors = ['#4e9af1', '#777']

axes[0].bar(quality['Region'], quality['PSNR (dB)'], color=colors, alpha=0.85)
axes[0].axhline(30, color='#e76f51', linestyle='--', linewidth=1, label='Target ≥ 30 dB')
axes[0].set_ylabel('PSNR (dB)', fontsize=10)
axes[0].set_title('PSNR — foreground vs. background', fontsize=11)
axes[0].set_ylim(0, max(quality['PSNR (dB)']) * 1.15)
axes[0].grid(axis='y', alpha=0.3)
axes[0].legend(fontsize=8)
for i, v in enumerate(quality['PSNR (dB)']):
    axes[0].text(i, v + 0.6, f'{v:.1f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

axes[1].bar(quality['Region'], quality['SSIM'], color=colors, alpha=0.85)
axes[1].axhline(0.85, color='#e76f51', linestyle='--', linewidth=1, label='Target ≥ 0.85')
axes[1].set_ylabel('SSIM', fontsize=10)
axes[1].set_title('SSIM — foreground vs. background', fontsize=11)
axes[1].set_ylim(0, 1.05)
axes[1].grid(axis='y', alpha=0.3)
axes[1].legend(fontsize=8)
for i, v in enumerate(quality['SSIM']):
    axes[1].text(i, v + 0.02, f'{v:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.suptitle('Quality metrics on CDnet 2014 baseline', fontsize=12, y=1.03)
plt.tight_layout()
fig.savefig(RESULTS_DIR / 'fig_quality_metrics.png', bbox_inches='tight')
plt.show()
display(quality)

---
## 4. Compression by Scene Category

Per-scene-category compression ratio for Mode 0 vs. Mode 1. Lower-activity scenes (office, parking) compress more than high-activity scenes (highway, crowd) because Mode 1 can drop most of the timeline once motion stops.


In [ ]:
CATEGORY_DATA = [
    # (category, mode0_ratio, mode1_ratio)  — measured on CDnet baseline clips
    ('baseline/highway',         5.8,  8.1),
    ('baseline/office',          8.4, 14.2),
    ('baseline/pedestrians',     6.2,  9.7),
    ('nightVideos/bridgeEntry',  7.1, 12.3),
    ('nightVideos/busyBlvd',     5.3,  7.2),
    ('shadow/bungalows',         9.6, 16.8),
    ('cameraJitter/traffic',     4.9,  6.4),
    ('lowFramerate/port_0_17',  11.2, 19.4),
]

df_cat = pd.DataFrame(CATEGORY_DATA, columns=['Category', 'Mode 0', 'Mode 1'])
df_cat = df_cat.sort_values('Mode 1')

fig, ax = plt.subplots(figsize=(9, 5))
y = np.arange(len(df_cat))
ax.barh(y - 0.2, df_cat['Mode 0'], 0.35, label='Mode 0 (all frames)',  color='#4e9af1', alpha=0.85)
ax.barh(y + 0.2, df_cat['Mode 1'], 0.35, label='Mode 1 (motion only)', color='#50c878', alpha=0.85)
ax.set_yticks(y)
ax.set_yticklabels(df_cat['Category'], fontsize=9)
ax.set_xlabel('Compression ratio (x)', fontsize=11)
ax.set_title('Compression ratio by CDnet scene category', fontsize=12)
ax.axvline(1, color='#999', linewidth=0.8, linestyle='--')
ax.grid(axis='x', alpha=0.3)
ax.legend(fontsize=10)
plt.tight_layout()
fig.savefig(RESULTS_DIR / 'fig_compression_by_category.png', bbox_inches='tight')
plt.show()

---
## 5. Foreground Coverage by CDnet Category

The percentage of pixels labelled foreground by MOG2 — averaged across each CDnet category. Coverage averaging well below 10 % across every category is the empirical justification for the dual-CRF compression strategy: the vast majority of every frame is static background where heavy compression is safe.


In [ ]:
# From docs/final_report.md, table in section 7.
FG_COVERAGE = [
    ('turbulence',                1.57),
    ('badWeather',                1.67),
    ('lowFramerate',              3.07),
    ('thermal',                   3.17),
    ('intermittentObjectMotion',  3.31),
    ('dynamicBackground',         3.81),
    ('shadow',                    4.52),
    ('cameraJitter',              5.03),
    ('nightVideos',               5.66),
    ('baseline',                  8.11),
]

df_fg = pd.DataFrame(FG_COVERAGE, columns=['Category', 'Avg FG %'])

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(df_fg['Category'], df_fg['Avg FG %'], color='#50c878', alpha=0.85)
ax.set_xlabel('Avg foreground coverage (%)', fontsize=10)
ax.set_title('CDnet 2014 — average MOG2 foreground coverage by category', fontsize=11)
ax.axvline(10, color='#e76f51', linestyle='--', linewidth=1, label='10% threshold')
ax.grid(axis='x', alpha=0.3)
ax.legend(fontsize=9)
plt.tight_layout()
fig.savefig(RESULTS_DIR / 'fig_fg_coverage.png', bbox_inches='tight')
plt.show()

---
## 6. Storage Projection — 100 cameras × 60 days

Sponsor scenario from the March 23 kickoff: a Navy base with ~100 static cameras, retention requirement of 60 days. Naive H.264 puts the storage requirement at ~80 TB. The selective pipeline brings that down by 4–10× depending on the active mode.


In [ ]:
storage = pd.DataFrame([
    {'Mode': 'Naive H.264',        '1 day (GB/cam)': 13.5, '60 days × 100 cams (TB)': 81.0},
    {'Mode': 'Selective Mode 0',   '1 day (GB/cam)':  4.0, '60 days × 100 cams (TB)': 24.0},
    {'Mode': 'Selective Mode 1',   '1 day (GB/cam)':  1.0, '60 days × 100 cams (TB)':  4.0},
    {'Mode': 'Selective Mode 3',   '1 day (GB/cam)':  0.55,'60 days × 100 cams (TB)':  3.3},
])

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(storage))
ax.bar(x, storage['60 days × 100 cams (TB)'],
       color=['#999', '#4e9af1', '#50c878', '#e76f51'], alpha=0.85)
for i, v in enumerate(storage['60 days × 100 cams (TB)']):
    ax.text(i, v + 1.2, f'{v:.1f} TB', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(storage['Mode'], fontsize=9)
ax.set_ylabel('Storage required (TB)', fontsize=11)
ax.set_title('60-day storage projection — 100 cameras at 1080p30', fontsize=12)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
fig.savefig(RESULTS_DIR / 'fig_storage_projection.png', bbox_inches='tight')
plt.show()
display(storage)

---
## 7. Side-by-Side: Original vs. Mode 0 vs. Mode 1

A representative frame from a CDnet baseline clip rendered three ways:
the raw input, the Mode 0 encode (every frame at dual CRF), and the
Mode 1 encode (motion-gated frames only). When the underlying clip is not
available on disk the cell prints a friendly warning instead of raising,
so the notebook still runs end-to-end.

This figure is referenced from `docs/final_report.md` and used as a
stand-alone PNG in the capstone slide deck.

*ROADMAP item 3.6 — Bloodawn (KheivenD)*


In [ ]:
# Hunt for an input clip in this priority order:
#   1. data/samples/cdnet_mp4/baseline/highway.mp4 (the canonical demo clip)
#   2. any .mp4 under data/samples/cdnet_mp4/
#   3. any .mp4 under data/samples/
# If none is found we skip the figure rather than failing the notebook.
import importlib

def _find_demo_clip():
    candidates = [
        CDNET_DIR / 'baseline' / 'highway.mp4',
        CDNET_DIR / 'baseline' / 'pedestrians.mp4',
    ]
    for c in candidates:
        if c.exists():
            return c
    if CDNET_DIR.exists():
        for p in CDNET_DIR.rglob('*.mp4'):
            return p
    if SAMPLES_DIR.exists():
        for p in SAMPLES_DIR.rglob('*.mp4'):
            return p
    return None

clip = _find_demo_clip()

if clip is None:
    print('[!] No sample clip found under data/samples/. Skipping side-by-side figure.')
    print('    Drop a clip at data/samples/cdnet_mp4/baseline/highway.mp4 to enable.')
else:
    print(f'Using clip: {clip}')
    try:
        cv2 = importlib.import_module('cv2')
    except ImportError:
        cv2 = None

    if cv2 is None:
        print('[!] OpenCV not installed in this kernel. Skipping side-by-side figure.')
    else:
        cap = cv2.VideoCapture(str(clip))
        if not cap.isOpened():
            print(f'[!] Could not open {clip}. Skipping side-by-side figure.')
        else:
            # Skip past warmup so MOG2 has a real model.
            target_idx = 200
            for _ in range(target_idx):
                ok, frame = cap.read()
                if not ok:
                    break
            ok, frame = cap.read()
            cap.release()

            if not ok or frame is None:
                print(f'[!] Could not read frame {target_idx} from {clip}. Skipping.')
            else:
                # Build a Mode-0 view (annotated with green ROI boxes from MOG2)
                # and a Mode-1 view (motion-gated — show 'GATE OPEN' label when
                # foreground is present, blackout otherwise).
                from background_subtraction.background_subtraction import BackgroundSubtractor

                subtractor = BackgroundSubtractor(method='MOG2', var_threshold=50)
                cap2 = cv2.VideoCapture(str(clip))
                # Re-warm MOG2 by feeding warmup frames
                for _ in range(150):
                    ok2, fr2 = cap2.read()
                    if not ok2:
                        break
                    subtractor.apply(fr2)
                # Read the same target frame again to keep alignment with `frame`
                ok3, target_frame = cap2.read()
                cap2.release()
                if not ok3 or target_frame is None:
                    target_frame = frame.copy()

                mask = subtractor.apply(target_frame)
                regions = subtractor.get_foreground_regions(mask)
                has_motion = len(regions) > 0

                mode0 = target_frame.copy()
                for r in regions:
                    cv2.rectangle(mode0, (r.x, r.y), (r.x + r.w, r.y + r.h),
                                  (0, 255, 0), 2)

                mode1 = target_frame.copy() if has_motion else np.zeros_like(target_frame)
                label = 'MODE 1: GATE OPEN' if has_motion else 'MODE 1: SKIPPED'
                cv2.putText(mode1, label, (24, 36), cv2.FONT_HERSHEY_SIMPLEX,
                            0.85, (0, 200, 255) if has_motion else (90, 90, 90),
                            2, cv2.LINE_AA)

                # Convert BGR→RGB for matplotlib
                originals = [
                    ('Original', frame),
                    ('Mode 0 — dual CRF + ROI boxes', mode0),
                    ('Mode 1 — motion-gated', mode1),
                ]
                fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
                for ax, (name, img) in zip(axes, originals):
                    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
                    ax.set_title(name, fontsize=11)
                    ax.set_xticks([]); ax.set_yticks([])
                fig.suptitle(f'Side-by-side comparison — {clip.name}', fontsize=12, y=1.02)
                plt.tight_layout()
                fig.savefig(RESULTS_DIR / 'fig_side_by_side.png', bbox_inches='tight')
                plt.show()
                print('Saved:', RESULTS_DIR / 'fig_side_by_side.png')

---
## 8. Live Segments Database (optional)

If a `metadata.db` exists under `outputs/` this cell summarises the most
recent run — total segments, object types, total bytes saved. If no DB is
found the cell simply notes that and continues. This is the fastest way to
sanity-check that the pipeline has actually produced segments without
running it from this notebook.


In [ ]:
db_candidates = []
if OUTPUTS_DIR.exists():
    db_candidates = sorted(OUTPUTS_DIR.rglob('metadata.db'),
                           key=lambda p: p.stat().st_mtime, reverse=True)

if not db_candidates:
    print('No metadata.db found under outputs/. Run the pipeline (or run_gui.py) to generate one.')
else:
    db_path = db_candidates[0]
    print('Most recent DB:', db_path)
    con = sqlite3.connect(str(db_path))
    try:
        df_segs = pd.read_sql('SELECT * FROM segments ORDER BY timestamp DESC LIMIT 200', con)
    finally:
        con.close()

    print(f'Segments returned: {len(df_segs)}')
    if 'object_type' in df_segs.columns:
        print('\nObject types observed:')
        display(df_segs['object_type'].value_counts())
    if 'file_size' in df_segs.columns:
        total_mb = df_segs['file_size'].sum() / 1e6
        targets = int(df_segs.get('target_detected', pd.Series([0])).sum())
        print(f'\nTotal storage on disk: {total_mb:.1f} MB across {len(df_segs)} segments')
        print(f'Segments with detected targets: {targets}')
    display(df_segs.head(8))

---
## 9. Acceptance Criteria Summary

Final pass/fail status against the criteria documented in `docs/final_report.md` section 7.


In [ ]:
criteria = pd.DataFrame([
    {'Criterion': 'Compression ratio on test footage',
     'Target': '≥ 3x',
     'Result': '16.6x (background) / 6.3x (effective)',
     'Status': 'Met'},
    {'Criterion': 'PSNR on foreground ROIs',
     'Target': '≥ 30 dB',
     'Result': f"{HEADLINE['foreground_psnr_db']} dB",
     'Status': 'Met'},
    {'Criterion': 'SSIM on foreground ROIs',
     'Target': '≥ 0.85',
     'Result': f"{HEADLINE['foreground_ssim']}",
     'Status': 'Met'},
    {'Criterion': 'Pipeline runs end-to-end without errors',
     'Target': 'Pass',
     'Result': '265 tests passing on safe subset',
     'Status': 'Met'},
    {'Criterion': 'Open-source / royalty-free codec',
     'Target': 'Required',
     'Result': 'libx264 + libsvtav1 evaluated; H.264 default',
     'Status': 'Met'},
])

display(criteria.set_index('Criterion'))
criteria.to_csv(RESULTS_DIR / 'final_criteria.csv', index=False)
print('Saved:', RESULTS_DIR / 'final_criteria.csv')

---
## 10. Summary Table for Slide Deck

Single consolidated table — copy directly into the capstone slide deck appendix or `docs/final_report.md` if updates are needed.


In [ ]:
summary = pd.DataFrame([
    {'Mode'   : 'Mode 0 — All frames',
     'Median' : '6.3x',
     'PSNR'   : '41.2 dB / 29.1 dB (fg/bg)',
     'SSIM'   : '0.9783 / 0.7903 (fg/bg)',
     'Use'    : 'Max coverage · legal record'},
    {'Mode'   : 'Mode 1 — Motion only',
     'Median' : '9.3x',
     'PSNR'   : 'same (events only)',
     'SSIM'   : 'same (events only)',
     'Use'    : 'Active surveillance'},
    {'Mode'   : 'Mode 2 — BG keyframe',
     'Median' : '12.7x',
     'PSNR'   : 'foreground preserved',
     'SSIM'   : 'foreground preserved',
     'Use'    : 'Event + scene context'},
    {'Mode'   : 'Mode 3 — Object only',
     'Median' : '16.6x',
     'PSNR'   : 'object pixels preserved',
     'SSIM'   : 'object pixels preserved',
     'Use'    : 'Downstream CV pipeline'},
])

display(summary.set_index('Mode'))
summary.to_csv(RESULTS_DIR / 'final_summary_table.csv', index=False)
print('Saved:', RESULTS_DIR / 'final_summary_table.csv')

---

**Notebook authored by Bloodawn (KheivenD).**
Last updated: 2026-05-02 for the May 6 capstone.
For source data see `docs/final_report.md`, `docs/stress_test_results.md`, and the segments DB under `outputs/`.
